In [1]:
import csv
import os
import webbrowser

import jinja2

In [2]:
path = 'templates'
loader = jinja2.FileSystemLoader(searchpath = path)
env = jinja2.Environment(loader = loader)

In [3]:
def render(env, template_name, output_path, *args, **kwargs):
    template = env.get_template(template_name)
    rendered = template.render(*args, **kwargs)
    with open(output_path, 'w') as write_file:
        write_file.write(rendered)
    #webbrowser.open(output_path)

In [4]:
# main research page
data_path = 'data/research.tsv'
with open(data_path) as read_file:
    projects = list(csv.DictReader(read_file, delimiter='\t'))

projects[0]

{'description': 'Multiscale data integration using heterogeneous networks to predict drug indications',
 'image': '',
 'project_id': 'drug-repurposing',
 'short': 'Drug Repurposing',
 'title': 'Uncovering new uses for drugs'}

In [5]:
from matplotlib.colors import rgb2hex
import matplotlib.cm
from numpy import linspace
import random

title = "Dr. Daniel Himmelstein"
colors = matplotlib.cm.Spectral_r(linspace(0, 1, len(title)))
colors = list(map(rgb2hex, colors))
random.seed(0)
random.shuffle(colors)
shadow_info = list()
for letter, color in zip(title, colors):
    shadow_info.append({'letter': letter, 'color': color})

In [14]:
template_name = 'home.html'
output_path = 'output/index.html'
render(env, template_name, output_path, projects=projects, shadow_info=shadow_info)

In [7]:
template_name = 'research.html'
output_path = 'output/research/index.html'
render(env, template_name, output_path, carousel=projects)

In [8]:
# individual research projects
project_to_cards = dict()

data_path = 'data/research-cards.tsv'
with open(data_path) as read_file:
    reader = csv.DictReader(read_file, delimiter='\t')
    for card in reader:
        project_to_cards.setdefault(card['project_id'], []).append(card)

In [9]:
for project in projects:
    project_id = project['project_id']
    cards = project_to_cards.get(project_id, [])
    template_name = 'project.html'
    output_dir = os.path.join('output', 'research', project_id)
    if not os.path.isdir(output_dir):
        os.mkdir(output_dir)
    output_path = os.path.join(output_dir, 'index.html')
    render(env, template_name, output_path, projects = projects, project = project,
           project_id = project_id, cards = cards)